# Mini Research Problem: Comparing SFT+LoRA and GRPO+LoRA for Local Math Reasoning

## Combined notebook

This notebook combines two project paths into one **mini research problem (MRP)** narrative:

1. **Supervised Fine-Tuning (SFT) + LoRA** with a rank sweep: **r = 8 / 4 / 2**
2. **GRPO + LoRA** with **r = 2** as a reinforcement-learning-style comparison


### Main idea

Inspired by **“Learning to Reason in 13 Parameters,”** this project studies a practical local question:

> **How far can compact parameter-efficient tuning improve math reasoning in a notebook-scale environment, and how do SFT+LoRA and GRPO+LoRA compare under similar constraints?**

This notebook also includes a **plausible exploration path** for **LoRA-XS** and **TinyLoRA** as future extensions, but does **not** claim completed empirical results for those methods unless you add them later.


## 0. Notebook organization

This notebook is organized as a mini research problem:

1. Motivation and research questions  
2. Dataset design and benchmark split  
3. Prompting and answer extraction  
4. Shared evaluation helpers  
5. Path A: **SFT + LoRA rank sweep**  
6. Path B: **GRPO + LoRA (r=2)**  
7. Comparison tables and visual analysis  
8. Error analysis and interpretation  
9. LoRA-XS / TinyLoRA exploration path  
10. Limitations and future work


## 1. Motivation

Large language models can solve many mathematical reasoning tasks, but **full fine-tuning is expensive**. Parameter-efficient tuning methods such as **LoRA** make local experimentation more realistic.

This project is motivated by two connected questions:

- Can **small LoRA updates** already improve local math reasoning performance?
- Does **reinforcement-learning-style post-training** with LoRA provide a different trade-off from **supervised fine-tuning**?

The project is also inspired by the paper **“Learning to Reason in 13 Parameters”**, which suggests that surprisingly small trainable parameter budgets may still recover meaningful reasoning gains. Rather than trying to replicate that paper exactly, this notebook uses it as a **design inspiration** for testing **compact adaptation paths** in a local environment.


## 2. Research questions and hypotheses

### Research questions

1. Can **SFT + LoRA** improve exact-answer math reasoning over the base model?
2. Which SFT LoRA rank among **8 / 4 / 2** gives the best efficiency–performance trade-off?
3. How does **GRPO + LoRA (r=2)** compare against **SFT + LoRA (r=2)** on the same evaluation format?
4. Are there signs that smaller adaptation methods such as **LoRA-XS** or **TinyLoRA** are promising next steps?

### Hypotheses

- **H1:** LoRA fine-tuning improves held-out exact-answer accuracy over the untuned base model.
- **H2:** Smaller ranks may already be competitive because the training budget and task are relatively focused.
- **H3:** GRPO + LoRA may improve response discipline and answer-format alignment, even if it is slower or harder to run locally.
- **H4:** LoRA-XS / TinyLoRA are plausible future directions for pushing parameter efficiency further, but should be treated as **exploratory extensions** unless fully implemented and evaluated.


In [1]:
import os
import re
import json
import math
import time
import random
from typing import Dict, List, Any, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("Environment ready.")


Environment ready.


## 3. Dataset design and benchmark split

This notebook uses different datasets for different roles.

### Training and validation
- `data/train_easy_math_120.jsonl`
- `data/val_easy_math_30.jsonl`

These are used for local tuning and validation.

### Evaluation / held-out benchmarks
- `data/benchmark_100.jsonl` → general math reasoning development benchmark
- `data/olympiad_style_100.jsonl` → more difficult olympiad-style problems
- `data/aimo_100.jsonl` → AIMO-style benchmark

In [2]:
TRAIN_PATH = "data/train_easy_math_120.jsonl"
VAL_PATH = "data/val_easy_math_30.jsonl"

REASONING_HELDOUT_PATH = "data/benchmark_100.jsonl"
OLYMPIAD_HELDOUT_PATH = "data/olympiad_style_100.jsonl"
AIMO_HELDOUT_PATH = "data/aimo_100.jsonl"

def load_jsonl(path):
    rows = []
    if not os.path.exists(path):
        print(f"Warning: file not found -> {path}")
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

benchmark_100 = load_jsonl(REASONING_HELDOUT_PATH)
olym_benchmark_100 = load_jsonl(OLYMPIAD_HELDOUT_PATH)
aimo_benchmark_100 = load_jsonl(AIMO_HELDOUT_PATH)

print("train:", len(train_data))
print("val:", len(val_data))
print("reasoning_100:", len(benchmark_100))
print("olympiad_100:", len(olym_benchmark_100))
print("aimo_100:", len(aimo_benchmark_100))


train: 120
val: 30
reasoning_100: 0
olympiad_100: 100
aimo_100: 100


## 4. Prompt format

All conditions use the same base prompt format. This keeps the evaluation fair across:

- base model
- SFT + LoRA-r8
- SFT + LoRA-r4
- SFT + LoRA-r2
- GRPO + LoRA-r2

The prompt is intentionally simple so the comparison focuses on **training strategy** rather than prompt engineering.


In [3]:
def make_prompt(question: str) -> str:
    return f'''You are a math assistant.
Return only the final numeric answer.
Do not explain.

Examples:
Question: 12 * 13
Answer: 156

Question: 25 + 17
Answer: 42

Question: 84 / 6 + 15
Answer: 29

Now answer:
Question: {question}
Answer:'''


## 5. Answer extraction and normalization

The evaluation focuses on **exact final numeric answer accuracy**.  
This intentionally simplifies scoring and makes the benchmark fast to run locally.


In [4]:

def extract_final_answer(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    first_line = lines[0]
    m = re.fullmatch(r"-?\d+", first_line)
    return m.group(0) if m else ""


## 6. Shared evaluation helpers

The helpers below are shared across SFT and GRPO conditions so the comparison is aligned.

### Metrics used

- **final_answer_accuracy**: majority-vote exact-answer accuracy across 3 runs  
- **consistency_across_3_runs**: how often the three runs agree  
- **avg_token_length**: approximate output length  
- **accuracy_gain**: improvement over base  
- **accuracy_gain_per_1k_params / per_1m_params**: simple efficiency views


In [5]:
def evaluate_dataset_multi_run(data, runner, n_runs=3):
    rows = []

    for ex in data:
        outputs = []
        preds = []
        lengths = []

        for _ in range(n_runs):
            raw = runner(ex["question"])
            if raw is None:
                raw = ""
            raw = str(raw).strip()

            pred = extract_final_answer(raw)

            outputs.append(raw)
            preds.append(pred)
            lengths.append(len(raw.split()))

        majority_pred = max(set(preds), key=preds.count) if preds else ""
        majority_correct = majority_pred == ex["answer"]
        consistency = len(set(preds)) == 1

        rows.append({
            "id": ex.get("id", ""),
            "difficulty": ex.get("difficulty", "unknown"),
            "topic": ex.get("topic", "unknown"),
            "question": ex["question"],
            "gold": ex["answer"],

            "raw_output_1": outputs[0] if len(outputs) > 0 else "",
            "raw_output_2": outputs[1] if len(outputs) > 1 else "",
            "raw_output_3": outputs[2] if len(outputs) > 2 else "",

            "pred_1": preds[0] if len(preds) > 0 else "",
            "pred_2": preds[1] if len(preds) > 1 else "",
            "pred_3": preds[2] if len(preds) > 2 else "",

            "correct_1": preds[0] == ex["answer"] if len(preds) > 0 else False,
            "correct_2": preds[1] == ex["answer"] if len(preds) > 1 else False,
            "correct_3": preds[2] == ex["answer"] if len(preds) > 2 else False,

            "majority_pred": majority_pred,
            "majority_correct": majority_correct,
            "avg_approx_token_len": sum(lengths) / len(lengths) if lengths else 0,
            "consistent_across_3_runs": consistency,
        })

    return pd.DataFrame(rows)

def summarize_metrics(
    df,
    condition_name,
    training_time_sec=0,
    trainable_params=0,
    total_params=None,
    trainable_percent=None,
    base_accuracy=None
):
    final_answer_accuracy = df["majority_correct"].mean()
    avg_token_length = df["avg_approx_token_len"].mean()
    consistency_rate = df["consistent_across_3_runs"].mean()

    accuracy_gain = None
    improvement_per_parameter = None
    accuracy_gain_per_1k_params = None
    accuracy_gain_per_1m_params = None

    if base_accuracy is not None and trainable_params not in (None, 0):
        accuracy_gain = final_answer_accuracy - base_accuracy
        improvement_per_parameter = accuracy_gain / trainable_params
        accuracy_gain_per_1k_params = accuracy_gain / (trainable_params / 1000.0)
        accuracy_gain_per_1m_params = accuracy_gain / (trainable_params / 1_000_000.0)

    return {
        "condition": condition_name,
        "final_answer_accuracy": final_answer_accuracy,
        "avg_token_length": avg_token_length,
        "consistency_across_3_runs": consistency_rate,
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_percent": trainable_percent,
        "accuracy_gain": accuracy_gain,
        "improvement_per_parameter": improvement_per_parameter,
        "accuracy_gain_per_1k_params": accuracy_gain_per_1k_params,
        "accuracy_gain_per_1m_params": accuracy_gain_per_1m_params,
    }

def round_summary_table(df):
    df = df.copy()
    for col in [
        "final_answer_accuracy",
        "avg_token_length",
        "consistency_across_3_runs",
        "accuracy_gain",
        "improvement_per_parameter",
        "accuracy_gain_per_1k_params",
        "accuracy_gain_per_1m_params",
        "training_time_sec",
        "trainable_percent",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(6)
    return df

def show_wrong_details(df, max_rows=10):
    wrong = df[~df["majority_correct"]].head(max_rows)
    print(f"Number of incorrect examples shown: {len(wrong)}")
    for _, row in wrong.iterrows():
        print("=" * 100)
        print(f"ID: {row['id']}")
        print(f"Difficulty: {row['difficulty']}")
        print(f"Topic: {row.get('topic', 'unknown')}")
        print(f"Question: {row['question']}")
        print(f"Run 1 response: {row['raw_output_1']}")
        print(f"Run 2 response: {row['raw_output_2']}")
        print(f"Run 3 response: {row['raw_output_3']}")
        print(f"Majority extracted answer: {row['majority_pred']}")
        print(f"Gold answer: {row['gold']}")


## 7. Path A — SFT + LoRA rank sweep

This is the main finished path from the earlier notebook.

### Experimental design
- base model: `Qwen/Qwen2.5-Math-1.5B`
- tuning style: **LoRA + SFT**
- ranks tested: **8 / 4 / 2**
- local training path: **MLX / MLX-LM style workflow**
- evaluation: 3-run majority-vote exact-answer accuracy

The rank sweep directly tests whether **smaller LoRA ranks** can remain competitive or even better under tight local budgets.


In [6]:

# YAML-style configs used for the SFT path (example generation)
configs = {
    "config_lora_r8_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r8_300
lora_parameters:
  rank: 8
  dropout: 0.0
  scale: 10.0
""",
    "config_lora_r4_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r4_300
lora_parameters:
  rank: 4
  dropout: 0.0
  scale: 10.0
""",
    "config_lora_r2_300.yaml": """model: Qwen/Qwen2.5-Math-1.5B
train: true
data: data/lora_easy_math_v3
train_type: lora
train_mode: sft
batch_size: 4
learning_rate: 1e-5
iters: 300
adapter_path: adapters/qwen_math_lora_r2_300
lora_parameters:
  rank: 2
  dropout: 0.0
  scale: 10.0
"""
}

list(configs.keys())


['config_lora_r8_300.yaml',
 'config_lora_r4_300.yaml',
 'config_lora_r2_300.yaml']

In [13]:

SFT_METADATA = {
    "lora_r8_300": {
        "training_method": "SFT+LoRA",
        "rank": 8,
        "training_time_sec": 439.34,
        "trainable_params": 9_232_000,
        "total_params": 1_544_259_072,
        "trainable_percent": 0.598,
        "adapter_path": "adapters/qwen_math_lora_r8_300",
    },
    "lora_r4_300": {
        "training_method": "SFT+LoRA",
        "rank": 4,
        "training_time_sec": 514.17,
        "trainable_params": 4_616_000,
        "total_params": 1_544_259_072,
        "trainable_percent": 0.299,
        "adapter_path": "adapters/qwen_math_lora_r4_300",
    },
    "lora_r2_300": {
        "training_method": "SFT+LoRA",
        "rank": 2,
        "training_time_sec": 555.20,
        "trainable_params": 2_308_000,
        "total_params": 1_544_259_072,
        "trainable_percent": 0.150,
        "adapter_path": "adapters/qwen_math_lora_r2_300",
    },
}



,training_method,rank,training_time_sec,trainable_params,total_params,trainable_percent,adapter_path
lora_r8_300,SFT+LoRA,8,439.34,9232000,1544259072,0.598,adapters/qwen_math_lora_r8_300
lora_r4_300,SFT+LoRA,4,514.17,4616000,1544259072,0.299,adapters/qwen_math_lora_r4_300
lora_r2_300,SFT+LoRA,2,555.2,2308000,1544259072,0.15,adapters/qwen_math_lora_r2_300


In [9]:
from mlx_lm import load, generate
model, tokenizer = load("Qwen/Qwen2.5-Math-1.5B")
lora_r8_250_model, lora_r8_250_tokenizer = load("Qwen/Qwen2.5-Math-1.5B", adapter_path=LORA_SFT_METADATA["lora_r8_250"]["adapter_path"])
lora_r4_250_model, lora_r4_250_tokenizer = load("Qwen/Qwen2.5-Math-1.5B", adapter_path=LORA_SFT_METADATA["lora_r4_250"]["adapter_path"])
lora_r2_250_model, lora_r2_250_tokenizer = load("Qwen/Qwen2.5-Math-1.5B", adapter_path=LORA_SFT_METADATA["lora_r2_250"]["adapter_path"])

def run_model_base(question: str) -> str:
    prompt = make_prompt(question)
    response = generate(model, tokenizer, prompt=prompt, max_tokens=20)
    return response.splitlines()[0].replace("<END>", "").strip()

def run_model_lora_r8(question: str) -> str:
    prompt = make_prompt(question)
    response = generate(lora_r8_250_model, lora_r8_250_tokenizer, prompt=prompt, max_tokens=20)
    return response.splitlines()[0].replace("<END>", "").strip()

def run_model_lora_r4(question: str) -> str:
    prompt = make_prompt(question)
    response = generate(lora_r4_250_model, lora_r4_250_tokenizer, prompt=prompt, max_tokens=20)
    return response.splitlines()[0].replace("<END>", "").strip()

def run_model_lora_r2(question: str) -> str:
    prompt = make_prompt(question)
    response = generate(lora_r2_250_model, lora_r2_250_tokenizer, prompt=prompt, max_tokens=20)
    return response.splitlines()[0].replace("<END>", "").strip()


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

## 8. Path B — GRPO + LoRA (r=2)

This path adapts the `RL+LoRa CPU.ipynb` notebook into the same MRP structure.

### Why GRPO here?

The goal is not to claim GRPO is universally better than SFT, but to test whether an RL-style post-training method:

- improves answer-format discipline,
- improves reward-aligned behavior,
- and changes the accuracy/efficiency trade-off relative to SFT.

### Why not PPO / TRPO here?

For this project, **GRPO** is the more practical choice because it aligns better with modern LLM post-training workflows and is easier to frame in the current TRL-style setup than classical policy optimization baselines such as TRPO. The notebook focuses on **one clean RL branch** instead of spreading effort across too many algorithms.


In [10]:

# GRPO + LoRA training template (adapted from RL+LoRa CPU.ipynb)
# Run this only in an environment with transformers/trl/peft/datasets installed and enough runtime budget.

GRPO_TEMPLATE = r'''
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ACCELERATE_USE_CPU"] = "true"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

import json
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

model_name = "Qwen/Qwen2.5-Math-1.5B"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

# add the same make_prompt, extract_final_answer, and reward function here
'''
print(GRPO_TEMPLATE)



import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ACCELERATE_USE_CPU"] = "true"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

import json
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

model_name = "Qwen/Qwen2.5-Math-1.5B"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

# add the same make_prompt, extract_final_answer, and reward function here



In [14]:

GRPO_METADATA = {
    "condition": "grpo_lora_r2_cpu",
    "training_method": "GRPO+LoRA",
    "rank": 2,
    "training_time_sec": 30838.75,
    "trainable_params": 272_384,
    "total_params": 1_543_986_688,
    "trainable_percent": 0.017642,
}
pd.DataFrame([GRPO_METADATA])


,condition,training_method,rank,training_time_sec,trainable_params,total_params,trainable_percent
0,grpo_lora_r2_cpu,GRPO+LoRA,2,30838.75,272384,1544259072,0.0176


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_name = "Qwen/Qwen2.5-Math-1.5B"
rl_adapter_path = "grpo_lora_qwen15b_math_cpu_out/checkpoint-120"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
rl_model = PeftModel.from_pretrained(base_model, rl_adapter_path)
rl_model.eval()

def run_model_grpo_r2(question: str) -> str:
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = rl_model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text
    return generated_text.strip()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 9. Compare and consolidated observed results from the current notebooks

The table below records the **observed SFT results** already present in the original notebook and a **GRPO comparison row** based on the GRPO notebook metadata.

### Important honesty note
- LoRA-XS and TinyLoRA are **not** treated as completed empirical results here.

This section combines:
- observed benchmark accuracy,
- SFT metadata,
- GRPO metadata,
so that we can generate many different views of the trade-offs.


In [15]:

method_meta_rows = [
    {
        "condition": "base",
        "training_method": "base",
        "rank": 0,
        "training_time_sec": 0.0,
        "trainable_params": 0,
        "total_params": 1_543_714_000,
        "trainable_percent": 0.0,
    },
    {
        "condition": "lora_r8_300",
        "training_method": "SFT+LoRA",
        "rank": 8,
        "training_time_sec": LORA_SFT_METADATA["lora_r8_300"]["training_time_sec"],
        "trainable_params": LORA_SFT_METADATA["lora_r8_300"]["trainable_params"],
        "total_params": LORA_SFT_METADATA["lora_r8_300"]["total_params"],
        "trainable_percent": LORA_SFT_METADATA["lora_r8_300"]["trainable_percent"],
    },
    {
        "condition": "lora_r4_300",
        "training_method": "SFT+LoRA",
        "rank": 4,
        "training_time_sec": LORA_SFT_METADATA["lora_r4_300"]["training_time_sec"],
        "trainable_params": LORA_SFT_METADATA["lora_r4_300"]["trainable_params"],
        "total_params": LORA_SFT_METADATA["lora_r4_300"]["total_params"],
        "trainable_percent": LORA_SFT_METADATA["lora_r4_300"]["trainable_percent"],
    },
    {
        "condition": "lora_r2_300",
        "training_method": "SFT+LoRA",
        "rank": 2,
        "training_time_sec": LORA_SFT_METADATA["lora_r2_300"]["training_time_sec"],
        "trainable_params": LORA_SFT_METADATA["lora_r2_300"]["trainable_params"],
        "total_params": LORA_SFT_METADATA["lora_r2_300"]["total_params"],
        "trainable_percent": LORA_SFT_METADATA["lora_r2_300"]["trainable_percent"],
    },
    {
        "condition": "grpo_lora_r2_cpu",
        "training_method": "GRPO+LoRA",
        "rank": RL_METADATA["rank"],
        "training_time_sec": RL_METADATA["training_time_sec"],
        "trainable_params": RL_METADATA["trainable_params"],
        "total_params": RL_METADATA["total_params"],
        "trainable_percent": RL_METADATA["trainable_percent"],
    },
]

METHOD_META = pd.DataFrame(method_meta_rows)

combined_summary = FINAL_RESULTS_OBSERVED.merge(
    METHOD_META,
    on=["condition", "training_method", "rank"],
    how="left"
)

base_acc_map = (
    combined_summary[combined_summary["condition"] == "base"]
    .set_index("benchmark")["final_answer_accuracy"]
    .to_dict()
)

combined_summary["base_accuracy_for_benchmark"] = combined_summary["benchmark"].map(base_acc_map)
combined_summary["accuracy_gain"] = combined_summary["final_answer_accuracy"] - combined_summary["base_accuracy_for_benchmark"]
combined_summary.loc[combined_summary["condition"] == "base", "accuracy_gain"] = np.nan

combined_summary["accuracy_gain_per_1m_params"] = np.where(
    combined_summary["trainable_params"].fillna(0) > 0,
    combined_summary["accuracy_gain"] / (combined_summary["trainable_params"] / 1_000_000.0),
    np.nan
)

combined_summary["runtime_min"] = combined_summary["training_time_sec"] / 60.0
combined_summary = round_summary_table(combined_summary)

combined_summary


NameError: name 'FINAL_RESULTS_OBSERVED' is not defined

## 11. Visual analysis

The goal here is to create **many complementary visuals** rather than relying on a single table.

Suggested interpretation angles:
- raw accuracy,
- accuracy gain over base,
- rank vs performance,
- parameter efficiency,
- runtime cost,
- Pareto trade-offs.


In [ ]:

def plot_bar_accuracy_by_benchmark(summary_df):
    df = summary_df.dropna(subset=["final_answer_accuracy"]).copy()
    benchmarks = list(df["benchmark"].dropna().unique())
    fig, axes = plt.subplots(len(benchmarks), 1, figsize=(10, 4 * len(benchmarks)), squeeze=False)

    for ax, bench in zip(axes.flatten(), benchmarks):
        sub = df[df["benchmark"] == bench].copy()
        ax.bar(sub["condition"], sub["final_answer_accuracy"])
        ax.set_title(f"Final Answer Accuracy — {bench}")
        ax.set_ylabel("accuracy")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()

plot_bar_accuracy_by_benchmark(combined_summary)


In [ ]:

def plot_accuracy_gain(summary_df):
    df = summary_df[
        (summary_df["condition"] != "base") &
        (~summary_df["accuracy_gain"].isna())
    ].copy()

    if len(df) == 0:
        print("No non-base accuracy gains available yet.")
    else:
        benchmarks = list(df["benchmark"].dropna().unique())
        fig, axes = plt.subplots(len(benchmarks), 1, figsize=(10, 4 * len(benchmarks)), squeeze=False)

        for ax, bench in zip(axes.flatten(), benchmarks):
            sub = df[df["benchmark"] == bench].copy()
            ax.bar(sub["condition"], sub["accuracy_gain"])
            ax.set_title(f"Accuracy Gain Over Base — {bench}")
            ax.set_ylabel("gain")
            ax.tick_params(axis="x", rotation=30)

        plt.tight_layout()
        plt.show()

plot_accuracy_gain(combined_summary)


In [ ]:

def plot_rank_vs_accuracy(summary_df, benchmark="reasoning_50"):
    df = summary_df[
        (summary_df["benchmark"] == benchmark) &
        (summary_df["training_method"].isin(["SFT+LoRA", "GRPO+LoRA"]))
    ].dropna(subset=["final_answer_accuracy"]).copy()

    if len(df) == 0:
        print("No data available for this plot yet.")
        return

    for method in df["training_method"].unique():
        sub = df[df["training_method"] == method].sort_values("rank")
        plt.plot(sub["rank"], sub["final_answer_accuracy"], marker="o", label=method)

    plt.title(f"Rank vs Accuracy — {benchmark}")
    plt.xlabel("LoRA rank")
    plt.ylabel("final_answer_accuracy")
    plt.xticks(sorted(df["rank"].dropna().unique()))
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

plot_rank_vs_accuracy(combined_summary, benchmark="reasoning_50")


In [ ]:

def plot_params_vs_accuracy(summary_df, benchmark="reasoning_50"):
    df = summary_df[
        (summary_df["benchmark"] == benchmark) &
        (summary_df["condition"] != "base")
    ].dropna(subset=["final_answer_accuracy", "trainable_params"]).copy()

    if len(df) == 0:
        print("No data available for this plot yet.")
        return

    plt.scatter(df["trainable_params"], df["final_answer_accuracy"])
    for _, row in df.iterrows():
        plt.annotate(row["condition"], (row["trainable_params"], row["final_answer_accuracy"]))
    plt.xscale("log")
    plt.title(f"Trainable Parameters vs Accuracy — {benchmark}")
    plt.xlabel("trainable_params (log scale)")
    plt.ylabel("final_answer_accuracy")
    plt.ylim(0, 1)
    plt.show()

plot_params_vs_accuracy(combined_summary, benchmark="reasoning_50")


In [ ]:

def plot_runtime_vs_accuracy(summary_df, benchmark="reasoning_50"):
    df = summary_df[
        (summary_df["benchmark"] == benchmark) &
        (summary_df["condition"] != "base")
    ].dropna(subset=["final_answer_accuracy", "runtime_min"]).copy()

    if len(df) == 0:
        print("No data available for this plot yet.")
        return

    plt.scatter(df["runtime_min"], df["final_answer_accuracy"])
    for _, row in df.iterrows():
        plt.annotate(row["condition"], (row["runtime_min"], row["final_answer_accuracy"]))
    plt.title(f"Runtime vs Accuracy — {benchmark}")
    plt.xlabel("training runtime (minutes)")
    plt.ylabel("final_answer_accuracy")
    plt.ylim(0, 1)
    plt.show()

plot_runtime_vs_accuracy(combined_summary, benchmark="reasoning_50")


In [ ]:

def plot_efficiency(summary_df):
    df = summary_df[(summary_df["condition"] != "base")].dropna(subset=["accuracy_gain_per_1m_params"]).copy()

    if len(df) == 0:
        print("No efficiency values available yet.")
        return

    df = df.sort_values(["benchmark", "accuracy_gain_per_1m_params"], ascending=[True, False])
    benchmarks = list(df["benchmark"].dropna().unique())

    fig, axes = plt.subplots(len(benchmarks), 1, figsize=(10, 4 * len(benchmarks)), squeeze=False)
    for ax, bench in zip(axes.flatten(), benchmarks):
        sub = df[df["benchmark"] == bench]
        ax.bar(sub["condition"], sub["accuracy_gain_per_1m_params"])
        ax.set_title(f"Accuracy Gain per 1M Trainable Params — {bench}")
        ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()

plot_efficiency(combined_summary)


In [ ]:

def plot_pareto(summary_df, benchmark="reasoning_50"):
    df = summary_df[
        (summary_df["benchmark"] == benchmark) &
        (summary_df["condition"] != "base")
    ].dropna(subset=["final_answer_accuracy", "trainable_params", "runtime_min"]).copy()

    if len(df) == 0:
        print("No data available for Pareto plot yet.")
        return

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(df["trainable_params"], df["final_answer_accuracy"], s=np.clip(df["runtime_min"], 1, None) * 8)

    for _, row in df.iterrows():
        ax.annotate(f"{row['condition']}\n{row['runtime_min']:.1f} min", (row["trainable_params"], row["final_answer_accuracy"]))

    ax.set_xscale("log")
    ax.set_xlabel("trainable_params (log scale)")
    ax.set_ylabel("final_answer_accuracy")
    ax.set_title(f"Pareto-style View: Params vs Accuracy (bubble ~ runtime) — {benchmark}")
    ax.set_ylim(0, 1)
    plt.show()

plot_pareto(combined_summary, benchmark="reasoning_50")


In [ ]:

def plot_method_summary_heatmap_like(summary_df):
    df = summary_df.pivot_table(
        index="condition",
        columns="benchmark",
        values="final_answer_accuracy",
        aggfunc="mean"
    )
    display(df.style.background_gradient(cmap="Blues"))

plot_method_summary_heatmap_like(combined_summary)


## 12. Optional detailed visual analysis from per-example outputs

If you evaluate and save per-example dataframes such as:
- `df_reason_base`
- `df_reason_r8`
- `df_reason_r4`
- `df_reason_r2`
- `df_reason_grpo_r2`

you can use the cells below to generate richer visuals:
- difficulty breakdown,
- topic breakdown,
- consistency breakdown,
- error overlap.


In [ ]:

def summarize_by_group(df, group_col):
    if df is None or len(df) == 0 or group_col not in df.columns:
        return pd.DataFrame()
    out = (
        df.groupby(group_col)
        .agg(
            accuracy=("majority_correct", "mean"),
            consistency=("consistent_across_3_runs", "mean"),
            avg_len=("avg_approx_token_len", "mean"),
            n=("id", "count"),
        )
        .reset_index()
        .sort_values("accuracy", ascending=False)
    )
    return out

# Example usage after evaluation:
# summarize_by_group(df_reason_r2, "topic")
# summarize_by_group(df_reason_r2, "difficulty")


In [ ]:

def compare_group_accuracy(group_tables: Dict[str, pd.DataFrame], group_col: str, metric="accuracy"):
    valid = {k: v for k, v in group_tables.items() if v is not None and len(v) > 0 and group_col in v.columns}
    if not valid:
        print("No valid grouped tables to compare.")
        return

    merged = None
    for name, df in valid.items():
        tmp = df[[group_col, metric]].rename(columns={metric: name})
        merged = tmp if merged is None else merged.merge(tmp, on=group_col, how="outer")

    merged = merged.fillna(np.nan).set_index(group_col)
    display(merged.style.background_gradient(cmap="Greens"))


## 13. Comparison interpretation guide

A useful way to read the current results is:

### SFT rank sweep
- On the current observed results, **LoRA-r2** is especially competitive on `reasoning_50` and `aimo_100`.
- **LoRA-r4** appears strongest on `olympiad_50`.
- This suggests that **smaller ranks can remain surprisingly effective**, which is consistent with the project motivation.

### GRPO + LoRA
- GRPO+LoRA-r2 is a meaningful comparison because it uses **far fewer trainable parameters** than the SFT LoRA runs.
- However, its local runtime is much larger in the recorded notebook metadata.
- This makes GRPO especially interesting for **parameter efficiency**, but not automatically best for **wall-clock efficiency**.

### Practical takeaway
A compact MRP conclusion can be framed as:

> **SFT+LoRA is the cleaner local baseline and delivers strong benchmark gains, while GRPO+LoRA is a plausible post-training comparison that may be attractive when parameter budget matters more than runtime budget.**


## 14. LoRA-XS and TinyLoRA as future exploration paths

This section is intentionally framed as **future work / plausible path exploration**, not as completed results.

### Why include them?

Because the paper inspiration points toward **ultra-compact adaptation**, LoRA-XS and TinyLoRA are natural follow-ups after the standard LoRA rank sweep.

### Plausible next-step research questions

1. Can **LoRA-XS** preserve a meaningful fraction of the SFT+LoRA gains with far fewer trainable parameters?
2. Can **TinyLoRA** work as a stronger compression-oriented baseline?
3. Does RL-style post-training remain stable when moving from standard LoRA to smaller variants?

### Suggested experimental sequence

1. Reproduce the current **SFT LoRA rank sweep**
2. Add **LoRA-XS** under the same prompt and evaluation protocol
3. Add **TinyLoRA** under the same benchmark suite
4. Compare:
   - final answer accuracy
   - runtime
   - trainable parameters
   - gain per million parameters
   - robustness across harder benchmarks

### Important reporting rule

Do **not** mix speculative LoRA-XS / TinyLoRA claims into the final results table unless you have actually run and measured them.


In [ ]:

FUTURE_METHODS_TEMPLATE = pd.DataFrame([
    {
        "condition": "lora_xs_placeholder",
        "training_method": "LoRA-XS",
        "rank": np.nan,
        "benchmark": "reasoning_50",
        "final_answer_accuracy": np.nan,
        "training_time_sec": np.nan,
        "trainable_params": np.nan,
        "trainable_percent": np.nan,
        "status": "planned / not yet measured",
    },
    {
        "condition": "tinylora_placeholder",
        "training_method": "TinyLoRA",
        "rank": np.nan,
        "benchmark": "reasoning_50",
        "final_answer_accuracy": np.nan,
        "training_time_sec": np.nan,
        "trainable_params": np.nan,
        "trainable_percent": np.nan,
        "status": "planned / not yet measured",
    },
])
FUTURE_METHODS_TEMPLATE


## 15. Error analysis

Use this section to inspect mistakes qualitatively after you run the detailed evaluation dataframes.

Recommended questions:
- Are errors mostly arithmetic, formatting, or reasoning-chain failures?
- Does GRPO reduce malformed outputs?
- Are SFT and GRPO making mistakes on the same problems?
- Do failures shift with difficulty or topic?


In [ ]:

# Example usage after evaluation:
# show_wrong_details(df_reason_r2, max_rows=8)
# show_wrong_details(df_reason_grpo_r2, max_rows=8)


## 16. Reproducibility checklist

Before final submission, make sure the notebook clearly records:

- model name used
- training dataset and benchmark dataset names
- prompt format
- answer extraction rule
- number of evaluation runs per condition
- LoRA ranks
- training runtime
- trainable parameter counts
- hardware / local environment notes
- which results are completed vs future work


## 17. Final conclusion draft

A concise conclusion you can adapt for your report or presentation:

> This mini research problem compared two local adaptation strategies for math reasoning: **SFT+LoRA** and **GRPO+LoRA**. The SFT rank sweep showed that compact LoRA fine-tuning can substantially improve exact-answer performance over the base model, with small ranks such as **r=2** and **r=4** remaining highly competitive. The GRPO+LoRA branch introduced a reinforcement-learning-style comparison that appears especially interesting from a **parameter-efficiency** perspective, although it is much more expensive in local runtime. Inspired by *Learning to Reason in 13 Parameters*, the project suggests that meaningful reasoning gains can be recovered with surprisingly small adaptation budgets, while also motivating future exploration of **LoRA-XS** and **TinyLoRA** as even more compact extensions.


## 18. Export helper

Use this if you want to save the summary table for your slides or report.


In [ ]:

# combined_summary.to_csv("combined_mrp_summary.csv", index=False)
# FINAL_RESULTS_OBSERVED.to_csv("final_results_observed.csv", index=False)
# METHOD_META.to_csv("method_metadata.csv", index=False)

print("Uncomment the export lines above if you want CSV outputs.")
